<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/03_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Obiettivo e Metodologia della Baseline Avanzata

In questa sezione definisco le prestazioni di riferimento (Baseline) utilizzando esclusivamente le informazioni note *prima* della prima mossa. Lo scopo è quantificare il potere predittivo dei metadati per stabilire un benchmark statistico e probabilistico rigoroso.
Per garantire la massima solidità metodologica, l'analisi segue questi passaggi:

**Pipeline di Standardizzazione**: Applico uno StandardScaler integrato in una *Pipeline di scikit-learn*. Questo passaggio normalizza le distribuzioni delle feature numeriche, garantendo una convergenza ottimale della Regressione Logistica ed evitando distorsioni legate alla diversa scala dei metadati.
Confronto tra Scenari (A vs B):
Scenario A: Il modello effettua le previsioni conoscendo unicamente il divario di abilità teorica.
Scenario B (Metadati Completi): Il modello utilizza tutte le informazioni numeriche pre-partita (differenza Elo, rating assoluti, tempo sul timer e validità per il punteggio).
   Entrambi gli scenari vengono valutati tramite Accuratezza, ROC-AUC e Log-Loss per misurare la reale calibrazione delle probabilità.
**Misurazione della Variabilità:** Applico una Cross-Validation stratificata a 5-Fold sul Train Set per estrarre l'accuratezza sotto forma di media e deviazione standard, isolando il rumore statistico del campione.
Analisi Teorica (De-standardizzazione): Estraggo il coefficiente della differenza Elo assegnato dal modello. Tramite un'operazione di de-standardizzazione matematica, riporto il coefficiente alla sua scala originale per poterlo confrontare direttamente con il valore teorico della formula di Arpad Elo (0.00576).

# Obiettivo e Metodologia della Baseline Avanzata

In questa sezione definisco le prestazioni di riferimento (Baseline) utilizzando esclusivamente le informazioni note prima della prima mossa. Lo scopo è quantificare il potere predittivo dei metadati per stabilire un benchmark statistico e probabilistico rigoroso prima di introdurre le feature posizionali.

Per garantire la massima solidità metodologica, l'analisi segue questi passaggi:

**Punto di Riferimento Minimo (Dummy Classifier):** Reinserisco un `DummyClassifier` come livello zero della validazione. Questo algoritmo funge da "pavimento" statistico, prevedendo sempre la classe maggioritaria (il Bianco) e restituendo una probabilità statica (AUC = 0.500), quantificando di fatto il totale affidamento al caso.
 **Pipeline di Standardizzazione:** Applico uno `StandardScaler` integrato in una Pipeline di scikit-learn. Questo passaggio normalizza le distribuzioni delle feature numeriche, garantendo una convergenza ottimale della Regressione Logistica ed evitando distorsioni legate alla diversa scala dei metadati.
 **Confronto tra Scenari (A vs B):**
  * **Scenario A:** Il modello effettua le previsioni conoscendo unicamente il divario di abilità teorica (`rating_diff`).
  * **Scenario B (Metadati Completi):** Il modello utilizza tutte le informazioni numeriche pre-partita (differenza Elo, rating assoluti, tempo sul timer e validità per il punteggio).
* **Misurazione della Variabilità:** Applico una Cross-Validation stratificata a 5-Fold sul Train Set per estrarre l'accuratezza sotto forma di media e deviazione standard, isolando il rumore statistico del campione.
* **Analisi Teorica (De-standardizzazione):** Estraggo il coefficiente della differenza Elo assegnato dal modello. Tramite un'operazione di de-standardizzazione matematica, riporto il coefficiente alla sua scala originale per poterlo confrontare direttamente con il valore teorico della formula di Arpad Elo.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings

# Disabilita i warning di convergenza per mantenere pulito l'output
warnings.filterwarnings('ignore')

print("1. VALUTAZIONE BASELINE AVANZATA E DUMMY CLASSIFIER\n")

soglie = [10, 20, 30, 40]
risultati = []

# Imposta la Cross-Validation a 5 fette
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Inizializza il modello logistico incapsulato in una Pipeline con StandardScaler
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

# Inizializza il Dummy Classifier (prevede sempre in base alla classe maggioritaria / probabilità a priori)
dummy = DummyClassifier(strategy='prior')

for N in soglie:
    # Carica i dataset filtrati per la soglia N
    df_train = pd.read_csv(f'train_features_N{N}.csv')
    df_test = pd.read_csv(f'test_features_N{N}.csv')

    # Definisce il target
    y_train = df_train['winner']
    y_test = df_test['winner']

    # Calcola la differenza Elo (Corretto l'operatore di sottrazione)
    df_train['rating_diff'] = df_train['white_rating'] - df_train['black_rating']
    df_test['rating_diff'] = df_test['white_rating'] - df_test['black_rating']

    # Definisce gli scenari
    features_b = ['rating_diff', 'white_rating', 'black_rating', 'rated', 'tempo_stimato_sec']

    X_train_a = df_train[['rating_diff']]
    X_test_a = df_test[['rating_diff']]

    X_train_b = df_train[features_b]
    X_test_b = df_test[features_b]

    # --- SCENARIO ZERO: DUMMY CLASSIFIER ---
    cv_res_dum = cross_validate(dummy, X_train_a, y_train, cv=cv, scoring='accuracy')
    cv_acc_dum_mean = cv_res_dum['test_score'].mean() * 100

    dummy.fit(X_train_a, y_train)
    y_prob_dum = dummy.predict_proba(X_test_a)[:, 1]
    acc_dum = accuracy_score(y_test, dummy.predict(X_test_a))
    roc_dum = roc_auc_score(y_test, y_prob_dum)

    # --- SCENARIO A: Solo Rating Diff ---
    cv_res_a = cross_validate(pipeline, X_train_a, y_train, cv=cv, scoring='accuracy')
    cv_acc_a_mean = cv_res_a['test_score'].mean() * 100
    cv_acc_a_std = cv_res_a['test_score'].std() * 100

    pipeline.fit(X_train_a, y_train)
    y_pred_a = pipeline.predict(X_test_a)
    y_prob_a = pipeline.predict_proba(X_test_a)[:, 1]

    acc_a = accuracy_score(y_test, y_pred_a)
    roc_a = roc_auc_score(y_test, y_prob_a)
    loss_a = log_loss(y_test, y_prob_a)

    # Estrae i pesi matematici solo al primo ciclo (N=10)
    if N == 10:
        scaler_a = pipeline.named_steps['scaler']
        model_a = pipeline.named_steps['logreg']

        # De-standardizza il coefficiente per renderlo confrontabile con la formula teorica
        coef_elo = model_a.coef_[0][0] / scaler_a.scale_[0]
        # Corretta la formula dell'intercetta
        intercetta_bianco = model_a.intercept_[0] - (model_a.coef_[0][0] * scaler_a.mean_[0] / scaler_a.scale_[0])

    # --- SCENARIO B: Tutti i Metadati Pre-Partita ---
    cv_res_b = cross_validate(pipeline, X_train_b, y_train, cv=cv, scoring='accuracy')
    cv_acc_b_mean = cv_res_b['test_score'].mean() * 100
    cv_acc_b_std = cv_res_b['test_score'].std() * 100

    pipeline.fit(X_train_b, y_train)
    y_pred_b = pipeline.predict(X_test_b)
    y_prob_b = pipeline.predict_proba(X_test_b)[:, 1]

    acc_b = accuracy_score(y_test, y_pred_b)
    roc_b = roc_auc_score(y_test, y_prob_b)
    loss_b = log_loss(y_test, y_prob_b)

    # Salva la riga riassuntiva
    risultati.append({
        'N': N,
        'Test_Acc_Dummy (%)': round(acc_dum * 100, 2),
        'CV_Train_A (%)': f"{cv_acc_a_mean:.1f} ± {cv_acc_a_std:.2f}",
        'Test_Acc_A (%)': round(acc_a * 100, 2),
        'Test_ROC_AUC_A': round(roc_a, 3),
        'Test_LogLoss_A': round(loss_a, 3),
        'Test_Acc_B (%)': round(acc_b * 100, 2),
        'Test_ROC_AUC_B': round(roc_b, 3)
    })

# Stampa la tabella
print(pd.DataFrame(risultati).to_string(index=False))

print("\n2. ANALISI MATEMATICA E CONCLUSIONI (N=10)")
print(f"Coefficiente Teorico Elo (Formula di Arpad Elo: ln(10)/400): 0.00576")
print(f"Coefficiente Calcolato (rating_diff):                        {coef_elo:.5f}")
print(f"Intercetta (Vantaggio Innato del Bianco):                    {intercetta_bianco:.5f}")

1. VALUTAZIONE BASELINE AVANZATA E DUMMY CLASSIFIER

 N  Test_Acc_Dummy (%) CV_Train_A (%)  Test_Acc_A (%)  Test_ROC_AUC_A  Test_LogLoss_A  Test_Acc_B (%)  Test_ROC_AUC_B
10               52.09    64.7 ± 0.40           65.15           0.714           0.616           64.98           0.713
20               51.83    64.3 ± 0.66           64.79           0.710           0.619           64.76           0.710
30               51.66    64.0 ± 0.87           64.61           0.705           0.623           64.48           0.705
40               51.32    63.5 ± 0.90           64.08           0.697           0.629           64.23           0.697

2. ANALISI MATEMATICA E CONCLUSIONI (N=10)
Coefficiente Teorico Elo (Formula di Arpad Elo: ln(10)/400): 0.00576
Coefficiente Calcolato (rating_diff):                        0.00382
Intercetta (Vantaggio Innato del Bianco):                    0.07523


### Osservazione sul Campione Statistico al variare di N

Analizzando i risultati della tabella per gli Scenari A e B, si osserva una lieve ma costante flessione delle metriche predittive (Accuracy e ROC-AUC) al crescere della soglia N.

È importante sottolineare che all'aumentare di N, il campione statistico analizzato subisce una mutazione strutturale. I dataset generati per soglie più elevate, infatti, trattengono esclusivamente le partite la cui durata supera le N semimosse, di conseguenza, il modello valutato a N=40 non sta analizzando la stessa popolazione di partite vista a N=10, ma un sottoinsieme specifico: si tratta di incontri mediamente più lunghi ed equilibrati, in cui i giocatori hanno superato la fase di apertura senza commettere errori fatali immediati. In un campione così scremato e bilanciato, il solo divario teorico (Elo) perde una frazione del suo potere predittivo, rendendo il compito di classificazione naturalmente più complesso.

### Osservazioni Metodologiche

- **Modello A vs B:** L'aggiunta di tutti i metadati pre-partita (Scenario B) non produce miglioramenti rispetto al solo `rating_diff` (Scenario A). Questo è un risultato in sé: la differenza di Elo assorbe la quasi totalità del potere predittivo pre-partita.

- **Ipotesi Coefficiente Elo:** Il coefficiente calcolato empiricamente risulta inferiore a quello teorico. Si ipotizza che questa discrepanza sia dovuta all'elevato rumore di fondo e alla varianza insita nei rating online (sistema Glicko-2, account nuovi o in ricalibrazione), che 'appiattiscono' la curva delle probabilità riducendo la certezza matematica del vantaggio.

### Osservazione sul Campione Statistico al variare di N
Analizzando i risultati per gli Scenari A e B, si osserva una lieve ma costante flessione delle metriche predittive (Accuracy e ROC-AUC) al crescere della soglia N.
È importante sottolineare che all'aumentare di N, il campione statistico analizzato subisce un cambiamento strutturale. I dataset generati per soglie più elevate, infatti, contengono esclusivamente le partite la cui durata supera le N semimosse. Si ipotizza che, di conseguenza, il modello valutato a N=40 non stia analizzando la stessa popolazione di partite vista a N=10, ma un sottoinsieme specifico: incontri mediamente più lunghi e strategicamente equilibrati, in cui i giocatori hanno superato l'apertura senza commettere errori fatali immediati. In un campione scremato e bilanciato in questo modo, è del tutto naturale che il solo divario teorico iniziale (Elo) perda una frazione del suo potere predittivo.

### Osservazioni Metodologiche e Matematiche

* **Modello A vs B:** L'aggiunta di tutti i metadati pre-partita (Scenario B) non produce miglioramenti rispetto al solo `rating_diff` (Scenario A). Questo è un risultato in sé: la differenza di Elo assorbe la quasi totalità del potere predittivo pre-partita, confermando lo Scenario A come la Baseline d'elezione per i successivi confronti.
* **Derivazione della Costante Teorica:** Il valore teorico di 0.00576 non è casuale, ma deriva dalla formula logistica originale ideata da Arpad Elo per calcolare il valore atteso. Matematicamente, tale coefficiente è dato dalla calcolo $\ln(10) / 400 \approx 0.005756$.
* **Ipotesi sul Coefficiente Elo Empirico:** Il coefficiente calcolato empiricamente dal nostro modello risulta leggermente inferiore a quello teorico. È quindi ragionevole ipotizzare che questa discrepanza sia dovuta all'elevato rumore di fondo e alla varianza intrinseca nei rating online (velocità del sistema Glicko-2, account nuovi o in fase di calibrazione rapida), i quali tendono ad "appiattire" empiricamente la curva delle probabilità reali rispetto all'aspettativa matematica ideale.
* **Interpretazione dell'Intercetta (Il Vantaggio del Bianco):** La destandardizzazione ha isolato un'intercetta di base pari a circa $0.075$. Inviando questo valore alla funzione sigmoide della Regressione Logistica, ricaviamo la probabilità di base $P = \frac{1}{1 + e^{-0.075}} \approx 0.5187$. Questo dato quantifica scientificamente il vantaggio del tratto: a perfetta parità di Elo ($x = 0$), il giocatore con il Bianco vanta matematicamente una probabilità di vittoria del **51.9%**.